## 第4章「選ばれる要因を推定する」— ノートブック共通セットアップ

ノートブック全体で使用するパスの設定、データフレーム（DataFrame）の表示スタイルの適用を行う。


In [1]:
import sys
from pathlib import Path

import matplotlib_fontja  # noqa: F401

for path in [Path.cwd(), *Path.cwd().parents]:
    if (path / "common" / "notebook_style.py").exists():
        sys.path.insert(0, str(path))
        break

    if (path / "src" / "common" / "notebook_style.py").exists():
        sys.path.insert(0, str(path / "src"))
        break

from common.notebook_style import apply_dataframe_style

apply_dataframe_style()

# 第4章　選ばれる要因を推定する


## 第4章、第3節、第1項のコード

### Polarsによるサンプルデータの生成と前処理

スマートフォンの選択を題材にしたコンジョイント分析のサンプルデータを Polars で生成し、モデルの推定に必要なロング形式へ整える。説明を簡単にするため、価格・バッテリー駆動時間・ブランドの少数の属性だけを使う。

#### 処理内容

1. **ライブラリのインポート**
   `polars`、`numpy` をインポートする。

2. **選択実験のデータの設計**
   回答者数 500 人、1 人あたり 8 選択課題、製品 A・B・C の 3 択を設定し、価格・バッテリー駆動時間・ブランドの属性値を乱数で割り当てる。この乱数設計はコードを理解するための単純な例である。実務では、属性の組み合わせの偏りを抑え、少ない質問でも必要な係数を安定して推定できるように調査票を設計する。

3. **効用の計算と選択のシミュレーション**
   データの生成に設定した係数（`true_beta_price = -0.000028`、`true_beta_battery_hours = 0.12`、ブランドごとの係数）から決定的効用を求め、ガンベル分布の誤差を加える。最も効用が高い製品を選択結果として記録する。すべての選択肢に共通する定数項は、選択課題の中で差が出ず、データから値を決められないため入れない。

4. **入力形式の確認**
   欠損値、IDの重複、課題ごとの選択数を確認した後、`df.head(9)` で先頭 9 行を表示する。


In [3]:
import numpy as np
import polars as pl

rng = np.random.default_rng(42)

n_respondents = 500
n_choice_sets = 8  # 1人あたりの選択課題数
products = ["A", "B", "C"]

# 各選択肢の属性を設計（価格・バッテリー駆動時間・ブランド）
price_levels = {
    "A": [69800, 79800, 89800],
    "B": [59800, 69800, 79800],
    "C": [79800, 89800, 99800],
}
battery_hours_levels = {"A": [9, 12, 15], "B": [9, 12], "C": [12, 15, 18]}
brand_premium = {
    "A": 0.0,
    "B": -0.3,
    "C": 0.5,
}  # データの生成に設定したブランドごとの係数

# データの生成に設定した係数（推定値との比較用）
true_beta_price = -0.000028  # 1円あたり
true_beta_battery_hours = 0.12

records = []
for resp in range(n_respondents):
    for cs in range(n_choice_sets):
        utilities = {}
        attrs = {}
        for prod in products:
            price_yen = rng.choice(price_levels[prod])
            battery_hours = rng.choice(battery_hours_levels[prod])
            v = (
                true_beta_price * price_yen
                + true_beta_battery_hours * battery_hours
                + brand_premium[prod]
            )
            utilities[prod] = v + rng.gumbel()
            attrs[prod] = {
                "price_yen": price_yen,
                "battery_hours": battery_hours,
            }

        chosen_prod = max(utilities, key=utilities.get)
        for prod in products:
            records.append(
                {
                    "respondent_id": resp,
                    "choice_set_id": cs,
                    "alternative_id": prod,
                    "chosen": int(prod == chosen_prod),
                    "price_yen": attrs[prod]["price_yen"],
                    "battery_hours": attrs[prod]["battery_hours"],
                    "brand": prod,
                }
            )

df = pl.DataFrame(records)

# 推定前に入力形式を確認する
required_columns = [
    "respondent_id",
    "choice_set_id",
    "alternative_id",
    "chosen",
    "price_yen",
    "battery_hours",
    "brand",
]
assert set(required_columns).issubset(df.columns)
assert df.select(pl.col("chosen").is_in([0, 1]).all()).item()
assert df.select(pl.any_horizontal(pl.all().is_null())).to_series().sum() == 0

task_check = df.group_by("respondent_id", "choice_set_id").agg(
    pl.len().alias("rows"),
    pl.col("alternative_id").n_unique().alias("alternatives"),
    pl.col("chosen").sum().alias("chosen_sum"),
)
invalid_tasks = task_check.filter(
    (pl.col("rows") < 2)
    | (pl.col("rows") != pl.col("alternatives"))
    | (pl.col("chosen_sum") != 1)
)
assert invalid_tasks.is_empty(), "選択課題の行数、ID、chosenを確認してください"

df.head(9)

respondent_id,choice_set_id,alternative_id,chosen,price_yen,battery_hours,brand
i64,i64,str,i64,i64,i64,str
0,0,"""A""",1,69800,15,"""A"""
0,0,"""B""",0,69800,12,"""B"""
0,0,"""C""",0,79800,12,"""C"""
0,1,"""A""",0,89800,15,"""A"""
0,1,"""B""",1,69800,9,"""B"""
0,1,"""C""",0,89800,15,"""C"""
0,2,"""A""",0,89800,12,"""A"""
0,2,"""B""",0,69800,9,"""B"""
0,2,"""C""",1,79800,15,"""C"""


## 第4章、第3節、第2項のコード

### Polars＋statsmodelsによる多項ロジット（MNL）の推定

本章で扱う多項ロジット（MNL）は、選択課題ごとに候補を比較するモデルである。前項で生成した選択実験のデータに `statsmodels` の `ConditionalLogit` を適用し、観測された選択が最も得られやすくなる係数を推定する。選択課題IDを作成し、価格を円から万円単位に変換して、ブランドB・Cのダミー変数を作る。この例では製品名とブランド名が一対一で対応するため、ブランドの係数には製品ごとの固定的な違いも含まれる。

#### 処理内容

1. **選択課題IDの生成**
   `respondent_id`と`choice_set_id`の組み合わせに連番を付け、一意な`task_id`を作成する。

2. **価格変数の単位変換**
   `price_yen / 10000` で価格を円から万円単位（`price_man`）に変換する。ブランド A を基準に `brand_B` と `brand_C` を作る。

3. **ConditionalLogitによる推定**
   説明変数 `["price_man", "battery_hours", "brand_B", "brand_C"]`、目的変数 `chosen`、グループ変数 `task_id` を指定して係数を推定する。すべての選択肢に共通する定数項は、選択課題の中で差が出ないため入れない。

4. **推定した係数の表示**
   各属性について推定した係数をループで出力する。


In [5]:
from statsmodels.discrete.conditional_models import ConditionalLogit

task_keys = (
    df.select("respondent_id", "choice_set_id")
    .unique(maintain_order=True)
    .with_row_index("task_id")
)
df_model = df.join(
    task_keys, on=["respondent_id", "choice_set_id"]
).with_columns(
    (pl.col("price_yen") / 10000).alias("price_man"),
    (pl.col("brand") == "B").cast(pl.Int8).alias("brand_B"),
    (pl.col("brand") == "C").cast(pl.Int8).alias("brand_C"),
)

pdf = df_model.to_pandas()

features = ["price_man", "battery_hours", "brand_B", "brand_C"]
X = pdf[features]
y = pdf["chosen"]
groups = pdf["task_id"]

model = ConditionalLogit(y, X, groups=groups)
result = model.fit(disp=False)

print("推定された係数:")
name_width = max(map(len, result.params.index))
for name, coef in result.params.items():
    print(f"  {name:>{name_width}}: {coef:.4f}")

推定された係数:
      price_man: -0.3118
  battery_hours: 0.1384
        brand_B: -0.3742
        brand_C: 0.4593


### statsmodelsによる信頼区間と$p$値の確認

$p$値は、モデルの仮定の下で「係数が0である」としたとき、今回のようなデータが得られにくいかを示す。因果関係を証明するものではない。ここでは説明を簡単にするため、各選択課題を独立とみなす。実際のデータでは、同じ回答者による繰り返しの回答を考慮する必要がある。


In [6]:
print(result.summary())

                  Conditional Logit Model Regression Results                  
Dep. Variable:                 chosen   No. Observations:                12000
Model:               ConditionalLogit   No. groups:                       4000
Log-Likelihood:               -3982.5   Min group size:                      3
Method:                          BFGS   Max group size:                      3
Date:                Sun, 13 Sep 2026   Mean group size:                   3.0
Time:                        01:57:40                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
price_man        -0.3118      0.026    -12.182      0.000      -0.362      -0.262
battery_hours     0.1384      0.009     14.758      0.000       0.120       0.157
brand_B          -0.3742      0.054     -6.877      0.000      -0.481      -0.268
brand_C           0.4593      0.053  

## 第4章、第3節、第3項のコード

### 推定結果からオッズ比と信頼区間を整理する

推定された係数を Polars のデータフレームにまとめ、オッズ比（`OR = exp(coef)`）、95%信頼区間、$p$値を一覧表として整理する。小さい$p$値は0と誤解されないように `<0.0001` と表示する。

#### 処理内容

1. **係数・信頼区間・$p$値の取得**
   `result.params`、`result.conf_int()`、`result.pvalues` から各値を取得する。

2. **データフレームの作成**
   属性名・係数・オッズ比・信頼区間の下限と上限・$p$値を Polars のデータフレームに格納し、数値を小数第4位に丸める。

3. **結果の表示**
   `summary_df` を Jupyter のデータフレーム表示機能で確認する。


In [7]:
import numpy as np
import polars as pl

# statsmodelsの推定結果 result から係数と信頼区間を取得
params = result.params  # 係数
conf_int = result.conf_int()  # 95%信頼区間
pvalues = result.pvalues
odds_ratio = np.exp(params)
odds_ratio_conf_int = np.exp(conf_int)

feature_names = ["価格（万円）", "駆動時間（時間）", "ブランドB", "ブランドC"]
p_value_display = ["<0.0001" if p < 0.0001 else f"{p:.4f}" for p in pvalues]

summary_df = pl.DataFrame(
    {
        "属性": feature_names,
        "係数": params.tolist(),
        "オッズ比": odds_ratio.tolist(),
        "95%下限": odds_ratio_conf_int.iloc[:, 0].tolist(),
        "95%上限": odds_ratio_conf_int.iloc[:, 1].tolist(),
        "p値": p_value_display,
    }
).with_columns(
    pl.col("係数", "オッズ比", "95%下限", "95%上限").round(4),
)

print(summary_df.to_pandas().to_string(index=False))

      属性      係数   オッズ比  95%下限  95%上限      p値
  価格（万円） -0.3118 0.7322 0.6963 0.7698 <0.0001
駆動時間（時間）  0.1384 1.1484 1.1275 1.1697 <0.0001
   ブランドB -0.3742 0.6878 0.6182 0.7652 <0.0001
   ブランドC  0.4593 1.5829 1.4275 1.7552 <0.0001


### 支払意思額（WTP）の算出

各属性の支払意思額（WTP）を $\text{WTP}_k = -\beta_k / \beta_{\text{price}}$ で計算し、属性が1単位改善したときに何円の価格差と釣り合うかを求める。これは実際の支払額そのものではない。価格の係数が0に近い場合は計算結果が不安定になるため、WTPを施策の判断に使わない。デルタ法とは、各係数が少し変わったときにWTPがどれだけ変わるかを微分で求め、係数の不確かさをWTPへ反映する方法である。

#### 処理内容

1. **WTPの計算**
   バッテリー駆動時間・ブランドB・ブランドCについて、属性の係数を価格の係数（万円単位）で割り、円単位に換算する。係数のばらつきと係数どうしの関係を使って、WTPの標準誤差と95%信頼区間をおおよそ求める。


In [9]:
beta_price = params["price_man"]  # price(万円)の係数
cov_params = result.cov_params().to_numpy()
param_names = params.index.tolist()

wtp_results = {}
for attr in ["battery_hours", "brand_B", "brand_C"]:
    coef = params[attr]
    wtp = -coef / beta_price  # 単位：万円

    # WTPが各係数の変化にどれだけ反応するかを微分で求める
    gradient = np.zeros(len(param_names))
    gradient[param_names.index(attr)] = -1 / beta_price
    gradient[param_names.index("price_man")] = coef / beta_price**2
    wtp_se = np.sqrt(gradient @ cov_params @ gradient)
    ci_lower, ci_upper = wtp - 1.96 * wtp_se, wtp + 1.96 * wtp_se

    wtp_results[attr] = (wtp, ci_lower, ci_upper)
    print(
        f"WTP({attr:13s}): {wtp * 10000:+.0f}円 "
        f"[95% CI: {ci_lower * 10000:+.0f}, {ci_upper * 10000:+.0f}]"
    )

WTP(battery_hours): +4438円 [95% CI: +3541, +5335]
WTP(brand_B      ): -12004円 [95% CI: -15033, -8975]
WTP(brand_C      ): +14732円 [95% CI: +11759, +17704]


### 回答者単位で不確かさを調べる

この章のシミュレーションでは選択課題を独立に作っている。実データで同じ回答者が複数の課題に答える場合は、回答者の全回答をまとめて抽出し直す。次のセルは時間がかかるため、必要なときに `RUN_RESPONDENT_BOOTSTRAP = True` として実行する。


In [10]:
RUN_RESPONDENT_BOOTSTRAP = False

if RUN_RESPONDENT_BOOTSTRAP:
    boot_rng = np.random.default_rng(123)
    respondent_ids = df.get_column("respondent_id").unique().to_numpy()
    bootstrap_params = []

    for _ in range(200):
        sampled_ids = boot_rng.choice(
            respondent_ids, len(respondent_ids), replace=True
        )
        boot_df = pl.concat(
            [
                df.filter(
                    pl.col("respondent_id") == respondent_id
                ).with_columns(pl.lit(draw_id).alias("sampled_respondent_id"))
                for draw_id, respondent_id in enumerate(sampled_ids)
            ]
        )
        boot_keys = (
            boot_df.select("sampled_respondent_id", "choice_set_id")
            .unique(maintain_order=True)
            .with_row_index("task_id")
        )
        boot_model = (
            boot_df.join(
                boot_keys, on=["sampled_respondent_id", "choice_set_id"]
            )
            .with_columns(
                (pl.col("price_yen") / 10000).alias("price_man"),
                (pl.col("brand") == "B").cast(pl.Int8).alias("brand_B"),
                (pl.col("brand") == "C").cast(pl.Int8).alias("brand_C"),
            )
            .to_pandas()
        )
        boot_result = ConditionalLogit(
            boot_model["chosen"],
            boot_model[features],
            groups=boot_model["task_id"],
        ).fit(disp=False)
        bootstrap_params.append(boot_result.params.to_numpy())

    bootstrap_ci = np.quantile(
        np.vstack(bootstrap_params), [0.025, 0.975], axis=0
    )
    print(dict(zip(features, bootstrap_ci.T)))

### 条件を変えたときの選択確率

製品Bを69,800円・12時間、製品Cを89,800円・15時間に固定し、製品Aの値下げと駆動時間の延長を比べる。ここで求めるのはA・B・Cの中での選択確率であり、市場全体の購入確率ではない。


In [11]:
def choice_prob(profiles):
    utility = profiles @ params[features].to_numpy()
    weight = np.exp(utility - utility.max())
    return weight / weight.sum()


base = np.array(
    [
        [7.98, 12, 0, 0],  # A：79,800円・12時間
        [6.98, 12, 1, 0],  # B：69,800円・12時間
        [8.98, 15, 0, 1],  # C：89,800円・15時間
    ]
)
price_down = base.copy()
price_down[0, 0] = 6.98
battery_up = base.copy()
battery_up[0, 1] = 15

p_base = choice_prob(base)[0]
p_price = choice_prob(price_down)[0]
p_battery = choice_prob(battery_up)[0]

scenario_df = pl.DataFrame(
    {
        "製品Aの条件": [
            "79,800円・12時間",
            "69,800円・12時間",
            "79,800円・15時間",
        ],
        "選択確率": [p_base, p_price, p_battery],
        "変更前との差": [0.0, p_price - p_base, p_battery - p_base],
    }
).with_columns(pl.col("選択確率", "変更前との差").mul(100).round(1))
scenario_df

製品Aの条件,選択確率,変更前との差
str,f64,f64
"""79,800円・12時間""",27.1,0.0
"""69,800円・12時間""",33.6,6.6
"""79,800円・15時間""",36.0,8.9
